# Preprocessing Pipeline: Climate-Socioeconomic Panel

**Objective**: Transform cleaned EDA data into modeling-ready format with engineered features, proper scaling, and time-aware train/test split.

**Inputs**: 
- `../data/processed/df_cleaned.csv` (output from EDA notebook)

**Outputs**:
- `X_train`, `X_test`, `y_train`, `y_test` (saved to `../data/processed/`)
- `scaler.pkl`, `feature_names.json` (saved to `../models/`)
- Engineered feature documentation

**Key Steps**:
1. Load cleaned data
2. Implement feature engineering (per-capita metrics, lags, ratios, interactions)
3. Handle scaling with RobustScaler
4. TimeSeriesSplit: train on 1900-2009, test on 2010-2023
5. Save artifacts for reproducibility

**Note**: All transformations must be fit on training data only to avoid leakage.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import os
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


DATA_CLEAN = '../data/processed/df_cleaned.csv'
DATA_PROCESSED = '../data/processed/'
MODELS_DIR = '../models/'
FIGURES_DIR = '../outputs/figures/'

for dir_path in [DATA_PROCESSED, MODELS_DIR, FIGURES_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("Imports and paths configured successfully.")

Imports and paths configured successfully.


In [7]:
# LOAD DATA & INITIAL SETUP 
df = pd.read_csv(DATA_CLEAN)
print("Loaded dataset info:")
print(f"Shape: {df.shape}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"Countries: {df['Country'].nunique()}")

TARGETS = ['Temperature_Anomaly', 'CO2_Emissions']

# Sort by Country and Year 
df = df.sort_values(['Country', 'Year']).reset_index(drop=True)

# Temporal split to prevent future data leakage
SPLIT_YEAR = 2010
df_train = df[df['Year'] < SPLIT_YEAR].copy()
df_test = df[df['Year'] >= SPLIT_YEAR].copy()

print(f"\nTemporal split (Year < {SPLIT_YEAR}):")
print(f"Train set: {len(df_train)} rows | Years: {df_train['Year'].min()}-{df_train['Year'].max()}")
print(f"Test set:  {len(df_test)} rows | Years: {df_test['Year'].min()}-{df_test['Year'].max()}")

Loaded dataset info:
Shape: (23797, 26)
Year range: 1900 - 2023
Countries: 195

Temporal split (Year < 2010):
Train set: 21106 rows | Years: 1900-2009
Test set:  2691 rows | Years: 2010-2023


In [8]:
#FEATURE ENGINEERING
import sys
sys.path.append('..')
print(f"Total columns after engineering: {df.shape[1]}")
print(f"Columns: {df.columns.tolist()}")

Total columns after engineering: 26
Columns: ['Country', 'Year', 'Air_Pollution_Index', 'Arctic_Ice_Extent', 'Average_Rainfall', 'Average_Temperature', 'Biodiversity_Index', 'CO2_Emissions', 'Deforestation_Rate', 'Energy_Consumption_Per_Capita', 'Extreme_Weather_Events', 'Forest_Area', 'Fossil_Fuel_Usage', 'GDP', 'Industrial_Activity', 'Methane_Emissions', 'Ocean_Acidification', 'Per_Capita_Emissions', 'Policy_Score', 'Population', 'Renewable_Energy_Usage', 'Sea_Level_Rise', 'Solar_Energy_Potential', 'Temperature_Anomaly', 'Urbanization', 'Waste_Management']


In [9]:
#  PREPARE FEATURES, SCALING, AND SAVE ARTIFACTS 

# 1. Define target and feature columns
TARGETS = ['Temperature_Anomaly', 'CO2_Emissions']
DROP_COLS = ['Country', 'Year'] + TARGETS 

# Re-split engineered data by year (ensures temporal alignment after feature creation)
df_train = df[df['Year'] < SPLIT_YEAR].copy()
df_test = df[df['Year'] >= SPLIT_YEAR].copy()

# Separate features (X) and targets (y)
X_train = df_train.drop(columns=DROP_COLS)
y_train = df_train[TARGETS]
X_test = df_test.drop(columns=DROP_COLS)
y_test = df_test[TARGETS]

# 2. Handle any remaining missing values (robust fallback)

train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# 3. Apply RobustScaler (fit ONLY on training data)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# 4. Verify and print summary
print("Data preparation complete:")
print(f"Train features: {X_train_scaled.shape}")
print(f"Test features:  {X_test_scaled.shape}")
print(f"Number of features used for modeling: {X_train_scaled.shape[1]}")
print(f"Targets: {TARGETS}")

# 5. Save artifacts for reproducibility and next notebook
X_train_scaled.to_csv(f'{DATA_PROCESSED}X_train.csv', index=False)
X_test_scaled.to_csv(f'{DATA_PROCESSED}X_test.csv', index=False)
y_train.to_csv(f'{DATA_PROCESSED}y_train.csv', index=False)
y_test.to_csv(f'{DATA_PROCESSED}y_test.csv', index=False)
joblib.dump(scaler, f'{MODELS_DIR}scaler.pkl')

print(f"\nArtifacts saved successfully to {DATA_PROCESSED} and {MODELS_DIR}")

Data preparation complete:
Train features: (21106, 22)
Test features:  (2691, 22)
Number of features used for modeling: 22
Targets: ['Temperature_Anomaly', 'CO2_Emissions']

Artifacts saved successfully to ../data/processed/ and ../models/


In [10]:
#  PREPARE FEATURES, SCALING, AND SAVE ARTIFACTS 
import sys
sys.path.append('..')  # Ensure src/ is in Python path
from src.feature_engineering import engineer_full_pipeline
from src.feature_engineering import select_features_by_importance,select_socioeconomic_features
from sklearn.preprocessing import RobustScaler
import joblib
import os

#Run full feature engineering pipeline on the cleaned dataframe
print("🔄 Running feature engineering pipeline...")
df = engineer_full_pipeline(df)

#Define targets and columns to exclude from feature set
TARGETS = ['Temperature_Anomaly', 'CO2_Emissions']
DROP_COLS = ['Country', 'Year'] + TARGETS

#Temporal split (strictly prevents future data leakage)
df_train = df[df['Year'] < SPLIT_YEAR].copy()
df_test = df[df['Year'] >= SPLIT_YEAR].copy()

#Separate features and targets
y_train = df_train[TARGETS]
y_test = df_test[TARGETS]
X_train_raw = df_train.drop(columns=DROP_COLS)
X_test_raw = df_test.drop(columns=DROP_COLS)
selected_features = select_features_by_importance(X_train_raw, y_train, n_features=20)


#Feature selection (reduces to 10 high-signal features)
X_train_sel = X_train_raw[selected_features].copy()
X_test_sel = X_test_raw[selected_features].copy()

#Handle missing values (fit statistics on TRAIN only to prevent leakage)
train_medians = X_train_sel.median()
X_train_clean = X_train_sel.fillna(train_medians)
X_test_clean = X_test_sel.fillna(train_medians)

#Scaling (fit ONLY on training data, transform both)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_clean),
    columns=X_train_clean.columns,
    index=X_train_clean.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_clean),
    columns=X_test_clean.columns,
    index=X_test_clean.index
)

#Verification & Summary
print("\n✅ Data preparation complete:")
print(f"Train features: {X_train_scaled.shape}")
print(f"Test features:  {X_test_scaled.shape}")
print(f"Final features used for modeling: {list(X_train_scaled.columns)}")

#Save Artifacts for Notebook 3
os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

X_train_scaled.to_csv(f'{DATA_PROCESSED}X_train.csv', index=False)
X_test_scaled.to_csv(f'{DATA_PROCESSED}X_test.csv', index=False)
y_train.to_csv(f'{DATA_PROCESSED}y_train.csv', index=False)
y_test.to_csv(f'{DATA_PROCESSED}y_test.csv', index=False)
joblib.dump(scaler, f'{MODELS_DIR}scaler.pkl')

print(f"\n💾 Artifacts saved to {DATA_PROCESSED} and {MODELS_DIR}")

🔄 Running feature engineering pipeline...
Starting feature engineering...
Added 16 temporal features. Dropped initial NaN periods.
Feature engineering complete. Final shape: (22822, 47)
Selected 20 features by importance:
  1. CO2_per_Capita: 0.3538
  2. Population: 0.1682
  3. Average_Temperature: 0.0337
  4. GDP: 0.0257
  5. Renewable_Energy_Usage: 0.0247
  6. Deforestation_Rate: 0.0210
  7. Biodiversity_Index: 0.0209
  8. Energy_Consumption_Per_Capita: 0.0209
  9. Solar_Energy_Potential: 0.0194
  10. Sea_Level_Rise: 0.0182
  11. Policy_Score: 0.0157
  12. Urbanization: 0.0156
  13. Waste_Management: 0.0154
  14. Air_Pollution_Index: 0.0141
  15. Arctic_Ice_Extent: 0.0135
  16. Ocean_Acidification: 0.0127
  17. Average_Rainfall: 0.0123
  18. Renewable_Energy_Usage_lag1: 0.0106
  19. Extreme_Weather_Events: 0.0102
  20. Fossil_Fuel_Usage: 0.0099

✅ Data preparation complete:
Train features: (20131, 20)
Test features:  (2691, 20)
Final features used for modeling: ['CO2_per_Capita', 'Po